# Messy E-Commerce Sales Dataset — Veri Temizleme ve Analiz

**Veri seti:** [Messy E-Commerce Sales Dataset — Kaggle](https://www.kaggle.com/datasets/kandeelai22/messy-e-commerce-sales-dataset)  

## Proje Amacı
Gerçek dünyada sıkça karşılaşılan veri kalitesi sorunlarını (eksik değerler, tutarsız formatlar, aykırı değerler, veri giriş hataları) tespit edip temizlemek; ardından temizlenmiş veriden iş içgörüleri çıkarmak.

---

## İçerik
1. Kütüphaneler ve veri yükleme
2. Kirlilik haritası — sorunların tespiti
3. Veri temizleme (adım adım)
4. Temizlik öncesi / sonrası karşılaştırma
5. Analiz ve görselleştirme
6. Ana bulgular ve sonuç

## 1. Kütüphaneler ve Veri Yükleme

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Kütüphaneler yüklendi.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/messy_ecommerce_sales.csv')

print(f'Boyut: {df.shape}')
print('\n--- Sütunlar ve tipler ---')
print(df.dtypes)
df.head()

## 2. Kirlilik Haritası — Sorunların Tespiti

In [ ]:
# Sütun isimlerindeki boşlukları temizle
df.columns = df.columns.str.strip()
print('Sütunlar:', df.columns.tolist())

# Eksik değer haritası
print('\n--- Eksik Değerler ---')
eksik = df.isnull().sum()
eksik_yuzde = (df.isnull().sum() / len(df) * 100).round(2)
print(pd.DataFrame({'Eksik Adet': eksik, 'Yüzde (%)': eksik_yuzde})[eksik > 0])

# Quantity'deki sayısal olmayan değerler
print('\n--- Quantity: Sayısal Olmayan Değerler ---')
print(df[pd.to_numeric(df['Quantity'], errors='coerce').isnull()]['Quantity'].value_counts())

# Price'taki sayısal olmayan değerler
print('\n--- Price: Sayısal Olmayan Değerler ---')
print(df[pd.to_numeric(df['Price'], errors='coerce').isnull()]['Price'].value_counts())

# Duplicate kontrolü
print(f'\n--- Duplicate Satır Sayısı: {df.duplicated().sum()} ---')

# Kategorik tutarsızlıklar
print('\n--- Category Değerleri ---')
print(df['Category'].str.strip().value_counts())

### Tespit Edilen Sorunlar

| Sütun | Sorun | Detay |
|---|---|---|
| Sütun isimleri | Başında boşluk | `Customer_Name`, `Category` |
| `Category` | Tutarsız büyük/küçük harf | `electronics`, `ELECTRONICS`, `electronic`, `sports` |
| `Price` | Geçersiz metin değerleri | `abd` (×2), `four hundred`, `300$` |
| `Quantity` | Geçersiz değer | `4a` |
| `Category` | Eksik değer | %7.77 (8 satır) |
| `Price` | Eksik değer | %4.85 (5 satır) |
| `Quantity` | Eksik değer | %4.85 (5 satır) |
| `Total` | Eksik değer | %13.59 (14 satır) — Price/Quantity bozuk olduğu için |
| `Order_Date` | Object tipi | Datetime'a çevrilmeli |
| Tüm tablo | Duplicate | 1 tekrar eden satır |

## 3. Veri Temizleme (Adım Adım)

In [ ]:
# Orijinali koru
df_clean = df.copy()
print(f'Başlangıç: {len(df_clean)} satır')

# ADIM 1: Duplicate sil
df_clean = df_clean.drop_duplicates()
print(f'Duplicate silindi → {len(df_clean)} satır')

In [ ]:
# ADIM 2: Category tutarsızlıklarını açık map ile düzelt
# Not: .str.title() yeterli değil — ELECTRONICS gibi tüm büyük harfli değerleri
# doğru dönüştüremiyor. Açık eşleme daha güvenilir.
kategori_map = {
    'electronics': 'Electronics',
    'ELECTRONICS' : 'Electronics',
    'electronic'  : 'Electronics',
    'sports'      : 'Sports',
}
df_clean['Category'] = df_clean['Category'].str.strip().replace(kategori_map)
print('Category düzeltme sonrası:')
print(df_clean['Category'].value_counts())

In [ ]:
# ADIM 3: Price temizleme
# "300$" → "300" ($ işareti kaldır)
# "four hundred", "abd" → NaN (to_numeric ile otomatik)
df_clean['Price'] = (
    df_clean['Price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
)
df_clean['Price'] = pd.to_numeric(df_clean['Price'], errors='coerce')

# Eksik Price → medyan ile doldur (aykırı değerlere karşı dayanıklı)
price_medyan = df_clean['Price'].median()
df_clean['Price'] = df_clean['Price'].fillna(price_medyan)
print(f'Price medyanı: {price_medyan:.2f}')
print(f'Price NaN kaldı mı: {df_clean["Price"].isnull().sum()}')

In [ ]:
# ADIM 4: Quantity temizleme
# "4a" → NaN (to_numeric ile otomatik)
df_clean['Quantity'] = pd.to_numeric(df_clean['Quantity'], errors='coerce')

# Eksik Quantity → medyan ile doldur
qty_medyan = int(df_clean['Quantity'].median())
df_clean['Quantity'] = df_clean['Quantity'].fillna(qty_medyan)
df_clean['Quantity'] = df_clean['Quantity'].astype('Int64')  # nullable integer
print(f'Quantity medyanı: {qty_medyan}')
print(f'Quantity NaN kaldı mı: {df_clean["Quantity"].isnull().sum()}')

In [ ]:
# ADIM 5: Category eksik değerleri → mod ile doldur
category_mod = df_clean['Category'].mode()[0]
df_clean['Category'] = df_clean['Category'].fillna(category_mod)
print(f'Category modu: {category_mod}')

# ADIM 6: Total'i yeniden hesapla
# Price ve Quantity temizlendi, Total baştan hesaplanmalı
df_clean['Total'] = df_clean['Price'] * df_clean['Quantity']
print(f'Total NaN kaldı mı: {df_clean["Total"].isnull().sum()}')

In [ ]:
# ADIM 7: Order_Date → datetime
df_clean['Order_Date'] = pd.to_datetime(df_clean['Order_Date'], errors='coerce')
df_clean['Month']      = df_clean['Order_Date'].dt.month
df_clean['Month_Name'] = df_clean['Order_Date'].dt.strftime('%B')

# Parse edilemeyen 2 tarih kaydını raporla (ihmal edilebilir seviye)
print('Parse edilemeyen tarih satırları:')
print(df_clean[df_clean['Order_Date'].isnull()][['ID','Customer_Name']])

# FINAL KONTROL
print('\n══════════════════════════════')
print(f'Final veri seti: {len(df_clean)} satır, {len(df_clean.columns)} sütun')
kalan = df_clean.isnull().sum()
kalan_var = kalan[kalan > 0]
if len(kalan_var) > 0:
    print('Kalan eksik değerler (ihmal edilebilir):')
    print(kalan_var)
else:
    print('Eksik değer kalmadı!')
print('══════════════════════════════')

## 4. Temizlik Öncesi / Sonrası Karşılaştırma

| Sorun | Öncesi | Sonrası | Yöntem |
|---|---|---|---|
| Duplicate satır | 1 | 0 | `drop_duplicates()` |
| Category tutarsızlığı | 9 farklı yazım | 5 standart kategori | `replace()` map |
| Price geçersiz değer | 4 bozuk kayıt | 0 | `pd.to_numeric(errors='coerce')` + medyan |
| Quantity geçersiz değer | 1 bozuk kayıt | 0 | `pd.to_numeric(errors='coerce')` + medyan |
| Eksik Category | 8 satır | 0 | Mod ile doldurma |
| Eksik Price/Quantity | 5'er satır | 0 | Medyan ile doldurma |
| Eksik Total | 14 satır | 0 | Price × Quantity yeniden hesaplandı |
| Order_Date tipi | object | datetime64 | `pd.to_datetime()` |
| Sütun ismi boşlukları | `' Category'` | `'Category'` | `str.strip()` |

## 5. Analiz ve Görselleştirme

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('E-Commerce Satış Analizi — Temizlenmiş Veri', fontsize=14, fontweight='bold')

# 1. Kategoriye göre toplam satış
cat_sales = df_clean.groupby('Category')['Total'].sum().sort_values(ascending=True)
axes[0,0].barh(cat_sales.index, cat_sales.values, color='steelblue', alpha=0.85)
axes[0,0].set_title('Kategoriye Göre Toplam Satış')
axes[0,0].set_xlabel('Toplam Satış')
for i, v in enumerate(cat_sales.values):
    axes[0,0].text(v + 100, i, f'{v:,.0f}', va='center', fontsize=9)

# 2. Ödeme yöntemine göre sipariş dağılımı
odeme = df_clean['Payment_Method'].value_counts()
axes[0,1].pie(odeme.values, labels=odeme.index, autopct='%1.1f%%', startangle=90,
              colors=['#3498db','#e67e22','#2ecc71','#e74c3c'])
axes[0,1].set_title('Ödeme Yöntemine Göre Sipariş Dağılımı')

# 3. Sipariş durumuna göre toplam gelir
status_gelir = df_clean.groupby('Status')['Total'].sum().sort_values(ascending=True)
renkler_map = {'Delivered':'#2ecc71','Shipped':'#3498db',
               'Processing':'#e67e22','Cancelled':'#e74c3c','Returned':'#95a5a6'}
bar_colors = [renkler_map.get(s, '#bdc3c7') for s in status_gelir.index]
axes[1,0].barh(status_gelir.index, status_gelir.values, color=bar_colors, alpha=0.85)
axes[1,0].set_title('Sipariş Durumuna Göre Toplam Gelir')
axes[1,0].set_xlabel('Toplam Gelir')
for i, v in enumerate(status_gelir.values):
    axes[1,0].text(v + 100, i, f'{v:,.0f}', va='center', fontsize=9)

# 4. Aylara göre sipariş sayısı
ay_isimleri = {1:'Oca',2:'Şub',3:'Mar',4:'Nis',5:'May',6:'Haz',
               7:'Tem',8:'Ağu',9:'Eyl',10:'Eki',11:'Kas',12:'Ara'}
aylik = df_clean.dropna(subset=['Month'])
aylik_siparis = aylik.groupby('Month').size()
axes[1,1].bar(aylik_siparis.index, aylik_siparis.values, color='mediumpurple', alpha=0.85)
axes[1,1].set_xticks(list(ay_isimleri.keys()))
axes[1,1].set_xticklabels(list(ay_isimleri.values()))
axes[1,1].set_title('Aylara Göre Sipariş Sayısı')
axes[1,1].set_ylabel('Sipariş Sayısı')

plt.tight_layout()
plt.savefig('ecommerce_analiz.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sayısal özet
iptal_iade = df_clean['Status'].isin(['Cancelled','Returned']).sum()
print('═══════════════════════════════════════')
print('         ÖZET İSTATİSTİKLER')
print('═══════════════════════════════════════')
print(f'Toplam Gelir        : {df_clean["Total"].sum():>12,.2f}')
print(f'Ortalama Sipariş    : {df_clean["Total"].mean():>12,.2f}')
print(f'En Çok Satan Ürün   : {df_clean["Product"].value_counts().index[0]}')
print(f'En Yüksek Sipariş   : {df_clean["Total"].max():>12,.2f}')
print(f'İptal + İade Oranı  : %{(iptal_iade/len(df_clean)*100):.1f} ({iptal_iade} sipariş)')
print(f'İade Edilen Gelir   : {df_clean[df_clean["Status"]=="Returned"]["Total"].sum():>12,.2f}')
print('═══════════════════════════════════════')

## 6. Ana Bulgular ve Sonuç

| # | Bulgu | Detay |
|---|---|---|
| 1 | **Books en yüksek gelirli kategori** | 49.665 — Electronics'in 37 katı |
| 2 | **Electronics verisi güvenilmez** | Bozuk Price değerleri medyanla dolduruldu — bu kategorinin analizi daha kapsamlı veri temizliği gerektiriyor |
| 3 | **%41.2 iptal/iade oranı kritik** | Her 5 siparişten 2'si iade ya da iptal; gerçek bir iş sorunu |
| 4 | **İade edilen gelir 43.525** | En yüksek gelir kalemi "Returned" — teslim edilen (14.311) siparişten 3× fazla |
| 5 | **Nakit ödeme hâlâ dominant** | Cash on Delivery %32.4 — dijital ödeme alışkanlığı henüz tam oturmamış |

### Öğrenilen Veri Temizleme Teknikleri

| Teknik | Kullanım |
|---|---|
| `df.columns.str.strip()` | Sütun ismi boşluklarını temizleme |
| `pd.to_numeric(errors='coerce')` | Bozuk sayısal değerleri NaN'a çevirme |
| `.str.replace('$', '')` | Özel karakter temizleme |
| `.replace(dict)` | Tutarsız kategori eşleme |
| `.fillna(median)` | Sayısal eksik değerleri medyanla doldurma |
| `.fillna(mode)` | Kategorik eksik değerleri mod ile doldurma |
| `drop_duplicates()` | Tekrar eden satırları silme |
| `pd.to_datetime(errors='coerce')` | Tarih formatına güvenli dönüşüm |
| `astype('Int64')` | Nullable integer dönüşümü |

### Sonraki Adımlar
- İade nedeni analizi (ek veri gerektirir)
- Electronics kategorisinin ayrı, daha kapsamlı temizliği
- Ürün bazlı karlılık analizi